# Statystyka dla Data Analyst/Engineer — Moduł 14: Testy A/B w praktyce

W Modułach 6, 7 i 9 ANALIZOWALIŚMY już gotowe dane z testu A/B (`grupa_kampanii` i
`konwersja`). Ten moduł zajmuje się drugą stroną medalu: jak DOBRZE zaprojektować taki
test od samego początku (ile obserwacji potrzeba, jaką ma moc wykrywania efektu) i jak
uniknąć najczęstszych pułapek praktycznych, które psują wnioski nawet z poprawnie
policzonego testu.

## Spis treści
1. [Anatomia dobrego testu A/B](#sec1)
2. [Wielkość próby i moc statystyczna](#sec2)
3. [Ile obserwacji potrzeba?](#sec3)
4. [Jaką moc miał nasz test?](#sec4)
5. [Istotność statystyczna a praktyczna](#sec5)
6. [Pułapka: podglądanie wyników](#sec6)
7. [Pułapka: wielokrotne porównania](#sec7)
8. [Podsumowanie i ćwiczenia](#sec8)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Anatomia dobrego testu A/B

Dobrze zaprojektowany test A/B ma kilka cech, ustalonych PRZED zebraniem jakichkolwiek
danych:

1. **Losowy przydział do grup** — dokładnie to, co dawało nam przyczynowość (nie tylko
   korelację) w Module 11: obie grupy powinny mieć podobny rozkład wszystkich cech poza
   samym testowanym czynnikiem.
2. **Jedna, wcześniej ustalona metryka główna** (*primary metric*) — u nas `konwersja`.
   Sprawdzanie wielu metryk naraz i wybieranie tej, która "wyszła" istotna, to wersja
   problemu wielokrotnych porównań z Modułu 10 (wracamy do tego w sekcji 7).
3. **Z góry ustalona wielkość próby** (albo formalna reguła zatrzymania) — na podstawie
   analizy mocy statystycznej (sekcje 2-4), NIE decyzja podejmowana "na oko" w trakcie
   zbierania danych (sekcja 6).
4. **Z góry ustalony poziom istotności** `alpha` (zwykle `0.05`) i pożądana **moc**
   (zwykle `0.8`).

<a id="sec2"></a>
## 2. Wielkość próby i moc statystyczna

**Moc statystyczna** (*statistical power*) to prawdopodobieństwo wykrycia efektu, JEŚLI
on naprawdę istnieje — czyli `1 - beta`, gdzie `beta` to ryzyko błędu II rodzaju z
Modułu 7 (nieodrzucenie H0, mimo że H1 jest prawdziwa). Moc zależy od trzech rzeczy
naraz:

- **wielkości efektu** — im większa rzeczywista różnica między grupami, tym łatwiej ją
  wykryć,
- **wielkości próby** — więcej obserwacji, mniej szumu, łatwiej wykryć nawet mały efekt,
- **poziomu `alpha`** — im bardziej rygorystyczny próg istotności, tym trudniej coś
  wykryć przy tej samej próbie.

Dla porównania dwóch proporcji (dokładnie nasz przypadek: `konwersja` w grupie A vs B)
`statsmodels` dostarcza gotowe narzędzia do tej analizy.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

analiza_mocy = NormalIndPower()


<a id="sec3"></a>
## 3. Ile obserwacji potrzeba?

Zanim zbierzemy jakiekolwiek dane, możemy zapytać: "ile obserwacji na grupę potrzebuję,
żeby wykryć różnicę konwersji z `10%` do `20%`, z mocą `0.8` przy `alpha=0.05`?".

In [ ]:
wielkosc_efektu = proportion_effectsize(0.20, 0.10)
print(f"Wielkosc efektu (h Cohena dla proporcji): {wielkosc_efektu:.4f}")

n_potrzebne = analiza_mocy.solve_power(
    effect_size=wielkosc_efektu, alpha=0.05, power=0.8, ratio=1.0, alternative="two-sided"
)
print(f"Potrzebna liczba obserwacji na grupe: {n_potrzebne:.0f}")


> 📊 **proportion_effectsize liczy tzw. h Cohena**
>
> `proportion_effectsize` przekształca dwie proporkcje na jedną liczbę opisującą "odległość" między nimi na specjalnej, znormalizowanej skali (arcsine), niezależną od wielkości próby — dokładnie tak samo jak Cohen's d z Modułu 8 robił to dla różnicy średnich.

> ⚠️ **Mały efekt wymaga DUŻO większej próby**
>
> Wykrycie subtelnej różnicy wymaga znacznie więcej danych niż wykrycie dużej różnicy — sprawdzisz to sam w ćwiczeniu 1 poniżej dla różnicy `10%` -> `15%` zamiast `10%` -> `20%`. To dlatego drobne usprawnienia UX w dużych firmach testuje się na milionach użytkowników, a nie na kilkuset.

<a id="sec4"></a>
## 4. Jaką moc miał nasz test?

Możemy też zadać pytanie odwrotne: mając już przeprowadzony test (nasze `klienci`),
jaką faktycznie miał moc wykrycia zaobserwowanej różnicy?

In [ ]:
wskaznik_konwersji = klienci.groupby("grupa_kampanii")["konwersja"].agg(["mean", "count"])
print(wskaznik_konwersji)

wielkosc_efektu_obserwowana = proportion_effectsize(
    wskaznik_konwersji.loc["B", "mean"], wskaznik_konwersji.loc["A", "mean"]
)
moc_osiagnieta = analiza_mocy.solve_power(
    effect_size=wielkosc_efektu_obserwowana,
    nobs1=wskaznik_konwersji.loc["A", "count"],
    alpha=0.05,
    ratio=wskaznik_konwersji.loc["B", "count"] / wskaznik_konwersji.loc["A", "count"],
    alternative="two-sided",
)
print(f"Osiagnieta moc testu: {moc_osiagnieta:.3f}")


Moc powyżej `0.8` (konwencjonalny próg "wystarczającej" mocy) oznacza, że nasz test
miał solidną szansę wykryć efekt tej wielkości, gdyby naprawdę istniał — co pasuje do
tego, że rzeczywiście wyszedł istotny w Module 9.

<a id="sec5"></a>
## 5. Istotność statystyczna a praktyczna

Przypomnienie z Modułów 7 i 11: **istotność statystyczna** (p-wartość poniżej `alpha`) to
nie to samo, co **istotność praktyczna** (czy efekt jest wystarczająco duży, żeby się
biznesowo liczył). Przy bardzo dużej próbie nawet różnica konwersji z `10.0%` do `10.3%`
może wyjść "istotna statystycznie" — ale czy warto wdrażać zmianę dla `0.3` punktu
procentowego, to pytanie BIZNESOWE, nie statystyczne. Zawsze raportuj razem: p-wartość,
wielkość efektu (tu: różnicę punktów procentowych albo iloraz szans z Modułu 13) i
przedział ufności (Moduł 6) dla tej różnicy.

<a id="sec6"></a>
## 6. Pułapka: podglądanie wyników

**Podglądanie wyników** (*peeking*, *optional stopping*) to sprawdzanie p-wartości WIELE
razy w trakcie zbierania danych i zatrzymanie testu, gdy tylko wyjdzie istotny wynik.
Sprawdźmy symulacyjnie (dokładnie jak w Module 7), co to robi z realnym ryzykiem błędu I
rodzaju, gdy H0 jest PRAWDZIWA (grupy naprawdę identyczne):

In [ ]:
from scipy import stats

rng2 = np.random.default_rng(7)
liczba_symulacji = 2000
n_maks = 400
punkty_kontrolne = [50, 100, 150, 200, 250, 300, 350, 400]

falszywe_alarmy_raz = 0
falszywe_alarmy_podgladanie = 0

for _ in range(liczba_symulacji):
    # H0 prawdziwa: obie grupy pochodza z tego samego rozkladu
    dane_a = rng2.normal(0, 1, size=n_maks)
    dane_b = rng2.normal(0, 1, size=n_maks)

    # (a) jeden test na samym koncu
    _, p_koncowe = stats.ttest_ind(dane_a, dane_b)
    if p_koncowe < 0.05:
        falszywe_alarmy_raz += 1

    # (b) test po kazdym punkcie kontrolnym, zatrzymanie przy pierwszym istotnym wyniku
    for k in punkty_kontrolne:
        _, p_k = stats.ttest_ind(dane_a[:k], dane_b[:k])
        if p_k < 0.05:
            falszywe_alarmy_podgladanie += 1
            break

print(f"Jednorazowy test na koncu: {falszywe_alarmy_raz / liczba_symulacji:.3f} falszywych alarmow")
print(f"Wielokrotne podgladanie ({len(punkty_kontrolne)} punktow): {falszywe_alarmy_podgladanie / liczba_symulacji:.3f} falszywych alarmow")


> ⚠️ **Podglądanie potrafi ponad trzykrotnie zawyżyć realne ryzyko błędu I rodzaju**
>
> Zauważ: przy H0 prawdziwej pojedynczy test na końcu daje realne ryzyko bliskie deklarowanym `5%` (`alpha=0.05`) — dokładnie zgodnie z definicją z Modułu 7. Ale sprawdzanie wyniku przy każdym z ośmiu punktów kontrolnych i zatrzymanie się na pierwszym istotnym wyniku podnosi realne ryzyko do mniej więcej `18%` — to ten sam problem wielokrotnych porównań z Modułu 10, tylko w czasie zamiast w przestrzeni zmiennych. Rozwiązania: ustal wielkość próby z góry (sekcja 3) i trzymaj się jej, albo użyj formalnej metody sekwencyjnej z korektą (np. korekta Bonferroniego -- zobacz wyzwanie poniżej).

<a id="sec7"></a>
## 7. Pułapka: wielokrotne porównania

To dokładnie ten sam problem z sekcji 1 Modułu 10, tylko zastosowany do testów A/B: jeśli
sprawdzamy WIELE metryk naraz (konwersja, satysfakcja, czas na stronie, wartość koszyka,
...) i uznajemy test za "sukces", gdy CHOĆ JEDNA z nich wyjdzie istotna, realne ryzyko
fałszywego alarmu rośnie dokładnie tak samo jak przy wielu jednoczesnych t-testach.

In [ ]:
liczba_metryk = 5
ryzyko_falszywego_alarmu = 1 - (1 - 0.05) ** liczba_metryk
print(f"Ryzyko przynajmniej jednego falszywego alarmu przy {liczba_metryk} metrykach: {ryzyko_falszywego_alarmu:.3f}")


<a id="sec8"></a>
## 8. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- kluczowe elementy dobrze zaprojektowanego testu A/B (losowy przydział, jedna metryka
  główna, z góry ustalona wielkość próby),
- moc statystyczną i jej zależność od wielkości efektu, wielkości próby i `alpha`,
- obliczanie potrzebnej wielkości próby oraz osiągniętej mocy przez
  `statsmodels.stats.power`,
- różnicę między istotnością statystyczną a praktyczną,
- pułapkę podglądania wyników (optional stopping) i jej wpływ na realne ryzyko błędu I
  rodzaju,
- problem wielokrotnych porównań zastosowany do wielu metryk w teście A/B.

> 📝 **Ćwiczenie 1: Wielkość próby dla mniejszego efektu**
>
> Policz potrzebną wielkość próby na grupę, żeby wykryć różnicę konwersji z `10%` do `15%` (zamiast `10%` do `20%` z sekcji 3), przy tej samej mocy `0.8` i `alpha=0.05`. Porównaj z wynikiem z sekcji 3.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
wielkosc_efektu_mala = proportion_effectsize(0.15, 0.10)
n_potrzebne_mala = analiza_mocy.solve_power(
    effect_size=wielkosc_efektu_mala, alpha=0.05, power=0.8, ratio=1.0, alternative="two-sided"
)
print(f"Potrzebna liczba obserwacji na grupe (10% -> 15%): {n_potrzebne_mala:.0f}")
```

Wychodzi ok. 680 obserwacji na grupe -- ponad 3.5 raza wiecej niz ok. 195 potrzebnych do wykrycia wiekszej roznicy 10% -> 20% z sekcji 3. Mniejszy efekt = znacznie wieksza potrzebna proba, dokladnie jak w warn() powyzej.
</details>

> 📝 **Ćwiczenie 2: Moc testu przy mniejszej próbie**
>
> Policz osiągniętą moc dla efektu z sekcji 3 (`10%` vs `20%`), ale przy znacznie mniejszej próbie: `nobs1=50` na grupę. Porównaj z mocą `0.8`, którą zaplanowaliśmy.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
moc_przy_50 = analiza_mocy.solve_power(
    effect_size=wielkosc_efektu, nobs1=50, alpha=0.05, ratio=1.0, alternative="two-sided"
)
print(f"Moc przy 50 obserwacjach na grupe: {moc_przy_50:.3f}")
```
</details>

> 📝 **Ćwiczenie 3: Mniej punktów kontrolnych, mniejsze zawyżenie**
>
> Powtórz symulację podglądania z sekcji 6, ale tylko z DWOMA punktami kontrolnymi (np. `[200, 400]`) zamiast ośmiu. Czy zawyżenie realnego ryzyka błędu I rodzaju jest mniejsze niż z ośmioma punktami?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rng3 = np.random.default_rng(7)
falszywe_alarmy_2pkt = 0
for _ in range(liczba_symulacji):
    dane_a = rng3.normal(0, 1, size=n_maks)
    dane_b = rng3.normal(0, 1, size=n_maks)
    for k in [200, 400]:
        _, p_k = stats.ttest_ind(dane_a[:k], dane_b[:k])
        if p_k < 0.05:
            falszywe_alarmy_2pkt += 1
            break
print(f"2 punkty kontrolne: {falszywe_alarmy_2pkt / liczba_symulacji:.3f} falszywych alarmow")
```

Realne ryzyko z 2 punktami kontrolnymi wychodzi nizsze niz z 8 (ale wciaz wyzsze niz nominalne 5%) -- im wiecej razy 'podgladamy', tym wieksze zawyzenie ryzyka falszywego alarmu.
</details>

> 📝 **Ćwiczenie 4: Ryzyko fałszywego alarmu przy wielu metrykach**
>
> Policz ryzyko przynajmniej jednego fałszywego alarmu (sekcja 7) dla `10` jednocześnie sprawdzanych metryk zamiast `5`. O ile wzrasta ryzyko?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
liczba_metryk_10 = 10
ryzyko_10 = 1 - (1 - 0.05) ** liczba_metryk_10
print(f"Ryzyko przy 10 metrykach: {ryzyko_10:.3f}")
```
</details>

> 🔥 **Wyzwanie: korekta Bonferroniego dla podglądania wyników**
>
> Powtórz symulację podglądania z sekcji 6 (8 punktów kontrolnych), ale tym razem porównuj p-wartość z KORYGOWANYM progiem `alpha_bonferroni = 0.05 / liczba_punktow` zamiast zwykłego `0.05` (dokładnie ta sama logika korekty co dla wielokrotnych porównań w Module 10). Czy realne ryzyko błędu I rodzaju wraca w okolice nominalnych `5%`?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rng4 = np.random.default_rng(7)
alpha_bonferroni = 0.05 / len(punkty_kontrolne)
falszywe_alarmy_bonferroni = 0

for _ in range(liczba_symulacji):
    dane_a = rng4.normal(0, 1, size=n_maks)
    dane_b = rng4.normal(0, 1, size=n_maks)
    for k in punkty_kontrolne:
        _, p_k = stats.ttest_ind(dane_a[:k], dane_b[:k])
        if p_k < alpha_bonferroni:
            falszywe_alarmy_bonferroni += 1
            break

print(f"Alpha po korekcie Bonferroniego: {alpha_bonferroni:.5f}")
print(f"Falszywe alarmy z korekta: {falszywe_alarmy_bonferroni / liczba_symulacji:.3f}")
```

Skorygowany prog (ok. 0.00625 zamiast 0.05) sprowadza realne ryzyko falszywego alarmu z powrotem w okolice nominalnych 5% (a nawet nieco ponizej, bo korekta Bonferroniego jest z natury nieco konserwatywna). To dziala, ale kosztem wiekszej wymaganej wielkosci efektu, zeby cokolwiek wykryc przy kazdym pojedynczym punkcie kontrolnym -- w praktyce czesciej stosuje sie dedykowane metody testowania sekwencyjnego (poza zakresem tego kursu).
</details>

## Co dalej?

To już przedostatni moduł tego kursu! W **Module 15**, module KOŃCOWYM, połączymy
wszystko: statystykę opisową, testowanie hipotez, regresję i dobre praktyki testów A/B w
jedną kompletną, end-to-end analizę biznesowego pytania — od pierwszego spojrzenia na
dane, przez sformułowanie hipotezy, po finalny raport z wnioskami.